<a href="https://colab.research.google.com/github/halimAhtasham/DeepLearning/blob/main/11_Prediction_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Heart Disease Prediction — Phase 11
## Prediction Pipeline

This notebook turns the finalized machine-learning model into a reusable prediction pipeline.

**Final model:** a tuned K-Nearest Neighbors (KNN) Pipeline containing `StandardScaler` followed by KNN.

Tuned KNN parameters:
- `n_neighbors = 21`
- `p = 1`
- `weights = uniform`

> **Important:** This is an educational machine-learning project, not a medical diagnostic system. Model outputs must not be used as clinical diagnoses or treatment decisions.


## Table of Contents

1. [Notebook Objective](#1-notebook-objective)
2. [Project Paths](#2-project-paths)
3. [Import Required Libraries](#3-import-required-libraries)
4. [Load the Final Model](#4-load-the-final-model)
5. [Verify the Model Pipeline](#5-verify-the-model-pipeline)
6. [Define Feature Order](#6-define-feature-order)
7. [Understand the Input Format](#7-understand-the-input-format)
8. [Create a Sample Patient](#8-create-a-sample-patient)
9. [Make a Prediction](#9-make-a-prediction)
10. [Display Prediction Probability](#10-display-prediction-probability)
11. [Create a Reusable Prediction Function](#11-create-a-reusable-prediction-function)
12. [Test the Function](#12-test-the-function)
13. [Validate User Input](#13-validate-user-input)
14. [Interactive Prediction](#14-interactive-prediction)
15. [Batch Prediction from CSV](#15-batch-prediction-from-csv)
16. [Save Prediction Results](#16-save-prediction-results)
17. [Test Model Reusability](#17-test-model-reusability)
18. [Project Workflow Summary](#18-project-workflow-summary)
19. [Final Conclusion](#19-final-conclusion)


# 1. Notebook Objective

The previous notebooks trained, evaluated, tuned, selected, and interpreted our ML model.

Now we answer:

> **Can we use the saved final model to make predictions on new data?**

Workflow:

```text
New Patient Data
       ↓
Correct Feature Order
       ↓
Saved ML Pipeline
       ↓
StandardScaler
       ↓
Tuned KNN
       ↓
Class + Probability
```

Because `StandardScaler` is already inside the saved Pipeline, we do **not** manually scale new input data.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/Machine Learning/Projects/Heart Disease Prediction"

DATASET_PATH = PROJECT_PATH + "/Dataset"
MODEL_PATH = PROJECT_PATH + "/Models"
RESULT_PATH = PROJECT_PATH + "/Results"

print("Project path:", PROJECT_PATH)
print("Model path:", MODEL_PATH)
print("Result path:", RESULT_PATH)


Project path: /content/drive/MyDrive/Machine Learning/Projects/Heart Disease Prediction
Model path: /content/drive/MyDrive/Machine Learning/Projects/Heart Disease Prediction/Models
Result path: /content/drive/MyDrive/Machine Learning/Projects/Heart Disease Prediction/Results


# 3. Import Required Libraries

- `pandas` → organize input data
- `numpy` → numerical validation
- `joblib` → load the saved model
- `os` → check files and paths


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd


# 4. Load the Final Model

The final model from Phase 09 was saved as:

```text
final_heart_disease_model.pkl
```

We load that model rather than retraining it.


In [ ]:
FINAL_MODEL_PATH = MODEL_PATH + "/final_heart_disease_model.pkl"

if not os.path.exists(FINAL_MODEL_PATH):
    raise FileNotFoundError(
        f"Final model was not found at: {FINAL_MODEL_PATH}"
    )

final_model = joblib.load(FINAL_MODEL_PATH)

print("Final model loaded successfully.")


Final model loaded successfully.


# 5. Verify the Model Pipeline

The expected structure is:

```text
StandardScaler
      ↓
KNN
```

with:

```text
n_neighbors = 21
p = 1
weights = uniform
```


In [ ]:
print(final_model)


Pipeline(steps=[('scaler', StandardScaler()),
                ('model', KNeighborsClassifier(n_neighbors=21, p=1))])


In [ ]:
print("Pipeline steps:")
for step_name, step_object in final_model.named_steps.items():
    print(f"- {step_name}: {step_object}")


Pipeline steps:
- scaler: StandardScaler()
- model: KNeighborsClassifier(n_neighbors=21, p=1)


In [ ]:
knn_model = final_model.named_steps["model"]

print("KNN parameters:")
print("n_neighbors:", knn_model.n_neighbors)
print("p:", knn_model.p)
print("weights:", knn_model.weights)


KNN parameters:
n_neighbors: 21
p: 1
weights: uniform


# 6. Define Feature Order

The prediction input must contain the same 13 features used during training, in the same order.

The target column is excluded because it is what the model predicts.


In [ ]:
FEATURE_NAMES = [
    "age",
    "sex",
    "cp",
    "trestbps",
    "chol",
    "fbs",
    "restecg",
    "thalach",
    "exang",
    "oldpeak",
    "slope",
    "ca",
    "thal"
]

print("Number of input features:", len(FEATURE_NAMES))
for i, feature in enumerate(FEATURE_NAMES, start=1):
    print(f"{i}. {feature}")


Number of input features: 13
1. age
2. sex
3. cp
4. trestbps
5. chol
6. fbs
7. restecg
8. thalach
9. exang
10. oldpeak
11. slope
12. ca
13. thal


# 7. Understand the Input Format

| Feature | Meaning |
|---|---|
| `age` | Age |
| `sex` | Sex encoded according to the dataset |
| `cp` | Chest pain type |
| `trestbps` | Resting blood pressure |
| `chol` | Serum cholesterol |
| `fbs` | Fasting blood sugar indicator |
| `restecg` | Resting ECG result |
| `thalach` | Maximum heart rate achieved |
| `exang` | Exercise-induced angina indicator |
| `oldpeak` | ST depression induced by exercise |
| `slope` | Slope of peak exercise ST segment |
| `ca` | Number of major vessels |
| `thal` | Thal-related dataset category |

**Important:** Categorical variables must use the same numeric encoding as the original dataset.


# 8. Create a Sample Patient

The following is an **illustrative example only** to demonstrate the pipeline.


In [ ]:
sample_patient = {
    "age": 55,
    "sex": 1,
    "cp": 1,
    "trestbps": 130,
    "chol": 240,
    "fbs": 0,
    "restecg": 1,
    "thalach": 150,
    "exang": 0,
    "oldpeak": 1.0,
    "slope": 1,
    "ca": 0,
    "thal": 2
}

sample_df = pd.DataFrame(
    [sample_patient],
    columns=FEATURE_NAMES
)

sample_df


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,55,1,1,130,240,0,1,150,0,1.0,1,0,2


# 9. Make a Prediction

The saved Pipeline automatically performs scaling before KNN prediction.

```text
sample_df
   ↓
StandardScaler
   ↓
KNN
   ↓
prediction
```


In [ ]:
prediction = final_model.predict(sample_df)

predicted_class = int(prediction[0])

if predicted_class == 1:
    result = "Heart Disease"
else:
    result = "No Heart Disease"

print("Predicted class:", predicted_class)
print("Prediction result:", result)


Predicted class: 1
Prediction result: Heart Disease


# 10. Display Prediction Probability

KNN provides probabilities based on its neighbor voting.

These are **model probabilities**, not calibrated clinical risk probabilities.


In [ ]:
probabilities = final_model.predict_proba(sample_df)

print("Probability of class 0:", probabilities[0][0])
print("Probability of class 1:", probabilities[0][1])


Probability of class 0: 0.19047619047619047
Probability of class 1: 0.8095238095238095


In [ ]:
predicted_probability = probabilities[0][predicted_class]

print(f"Predicted class: {predicted_class}")
print(f"Model probability: {predicted_probability:.2%}")


Predicted class: 1
Model probability: 80.95%


For example, if the model outputs `0.62` for class 1, say:

> The model assigned a probability of 62% to class 1.

Do **not** interpret this as a medically validated 62% probability of disease.


# 11. Create a Reusable Prediction Function

This function accepts one patient's feature dictionary and returns the class and probabilities.


In [ ]:
def predict_heart_disease(patient_data, model=final_model):
    """Predict the target class for one new observation."""

    missing_features = [
        feature for feature in FEATURE_NAMES
        if feature not in patient_data
    ]

    if missing_features:
        raise ValueError(
            f"Missing required features: {missing_features}"
        )

    input_df = pd.DataFrame(
        [patient_data],
        columns=FEATURE_NAMES
    )

    predicted_class = int(model.predict(input_df)[0])
    probabilities = model.predict_proba(input_df)[0]

    predicted_label = (
        "Heart Disease"
        if predicted_class == 1
        else "No Heart Disease"
    )

    return {
        "predicted_class": predicted_class,
        "predicted_label": predicted_label,
        "probability_class_0": probabilities[0],
        "probability_class_1": probabilities[1],
        "predicted_probability": probabilities[predicted_class]
    }


# 12. Test the Function

Now prediction can be performed with a single function call.


In [ ]:
patient_2 = {
    "age": 45,
    "sex": 0,
    "cp": 2,
    "trestbps": 120,
    "chol": 200,
    "fbs": 0,
    "restecg": 1,
    "thalach": 165,
    "exang": 0,
    "oldpeak": 0.5,
    "slope": 2,
    "ca": 0,
    "thal": 2
}

prediction_result = predict_heart_disease(patient_2)

prediction_result


{'predicted_class': 1,
 'predicted_label': 'Heart Disease',
 'probability_class_0': np.float64(0.047619047619047616),
 'probability_class_1': np.float64(0.9523809523809523),
 'predicted_probability': np.float64(0.9523809523809523)}

In [ ]:
print("Prediction Result")
print("-" * 40)
print("Predicted class :", prediction_result["predicted_class"])
print("Predicted label :", prediction_result["predicted_label"])
print("Class 0 probability:",
      f"{prediction_result['probability_class_0']:.2%}")
print("Class 1 probability:",
      f"{prediction_result['probability_class_1']:.2%}")
print("Predicted probability:",
      f"{prediction_result['predicted_probability']:.2%}")


Prediction Result
----------------------------------------
Predicted class : 1
Predicted label : Heart Disease
Class 0 probability: 4.76%
Class 1 probability: 95.24%
Predicted probability: 95.24%


# 13. Validate User Input

A prediction system should reject incomplete or malformed input instead of silently producing a result.

This basic validation checks:

- required features
- unexpected features
- missing values
- numeric values

It does not perform clinical validation.


In [ ]:
def validate_patient_data(patient_data):
    """Perform basic structural validation for one patient."""

    missing_features = [
        feature for feature in FEATURE_NAMES
        if feature not in patient_data
    ]

    if missing_features:
        return False, f"Missing features: {missing_features}"

    unexpected_features = [
        feature for feature in patient_data
        if feature not in FEATURE_NAMES
    ]

    if unexpected_features:
        return False, f"Unexpected features: {unexpected_features}"

    for feature in FEATURE_NAMES:
        value = patient_data[feature]

        if value is None:
            return False, f"Missing value for: {feature}"

        if not isinstance(
            value,
            (int, float, np.integer, np.floating)
        ):
            return False, f"Non-numeric value for: {feature}"

        if pd.isna(value):
            return False, f"NaN value for: {feature}"

    return True, "Input is structurally valid."


In [ ]:
is_valid, message = validate_patient_data(patient_2)

print("Valid:", is_valid)
print("Message:", message)


Valid: True
Message: Input is structurally valid.


# 14. Interactive Prediction

The following function lets a user enter all 13 feature values in Colab.

Run `interactive_prediction()` only when you want to enter a new example manually.

The numeric encoding must match the original dataset.


In [ ]:
def get_numeric_input(feature_name):
    """Ask the user for one numeric feature value."""

    while True:
        try:
            return float(input(f"Enter {feature_name}: "))
        except ValueError:
            print("Please enter a numeric value.")


In [ ]:
def interactive_prediction():
    """Collect feature values and make one prediction."""

    print("=" * 60)
    print("Heart Disease ML Prediction")
    print("=" * 60)
    print("Use the same feature encoding as the original dataset.")
    print()

    patient = {}

    for feature in FEATURE_NAMES:
        patient[feature] = get_numeric_input(feature)

    is_valid, message = validate_patient_data(patient)

    if not is_valid:
        print("Input error:", message)
        return

    result = predict_heart_disease(patient)

    print()
    print("=" * 60)
    print("PREDICTION RESULT")
    print("=" * 60)
    print("Predicted class :", result["predicted_class"])
    print("Predicted label :", result["predicted_label"])
    print("Class 0 probability:",
          f"{result['probability_class_0']:.2%}")
    print("Class 1 probability:",
          f"{result['probability_class_1']:.2%}")
    print("Predicted class probability:",
          f"{result['predicted_probability']:.2%}")
    print()
    print("Educational ML output only — not a medical diagnosis.")


### To run the interactive predictor

```python
interactive_prediction()
```

Colab will ask for each feature one at a time.


# 15. Batch Prediction from CSV

A practical ML system may need to process many rows.

Suppose a new CSV contains the same 13 feature columns. We can generate predictions for all rows at once.


In [ ]:
def predict_from_dataframe(input_df, model=final_model):
    """Predict multiple observations from a DataFrame."""

    missing_columns = [
        feature for feature in FEATURE_NAMES
        if feature not in input_df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    model_input = input_df[FEATURE_NAMES].copy()

    predictions = model.predict(model_input)
    probabilities = model.predict_proba(model_input)

    output_df = input_df.copy()
    output_df["predicted_class"] = predictions
    output_df["probability_class_0"] = probabilities[:, 0]
    output_df["probability_class_1"] = probabilities[:, 1]

    return output_df


### Example batch

We'll use our two example patients.


In [ ]:
batch_input = pd.DataFrame(
    [sample_patient, patient_2],
    columns=FEATURE_NAMES
)

batch_input


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,55,1,1,130,240,0,1,150,0,1.0,1,0,2
1,45,0,2,120,200,0,1,165,0,0.5,2,0,2


In [ ]:
batch_predictions = predict_from_dataframe(batch_input)

batch_predictions


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,predicted_class,probability_class_0,probability_class_1
0,55,1,1,130,240,0,1,150,0,1.0,1,0,2,1,0.190476,0.809524
1,45,0,2,120,200,0,1,165,0,0.5,2,0,2,1,0.047619,0.952381


# 16. Save Prediction Results

The example batch predictions can be saved in the project's `Results` folder.


In [ ]:
BATCH_RESULT_PATH = RESULT_PATH + "/sample_batch_predictions.csv"

batch_predictions.to_csv(
    BATCH_RESULT_PATH,
    index=False
)

print("Prediction results saved to:")
print(BATCH_RESULT_PATH)


Prediction results saved to:
/content/drive/MyDrive/Machine Learning/Projects/Heart Disease Prediction/Results/sample_batch_predictions.csv


# 17. Test Model Reusability

A saved model should be loadable later without retraining.

We simulate a fresh session by loading the `.pkl` file again.


In [ ]:
reloaded_model = joblib.load(FINAL_MODEL_PATH)

print("Model successfully reloaded.")
print(reloaded_model)


Model successfully reloaded.
Pipeline(steps=[('scaler', StandardScaler()),
                ('model', KNeighborsClassifier(n_neighbors=21, p=1))])


In [ ]:
reloaded_prediction = reloaded_model.predict(sample_df)[0]
reloaded_probability = reloaded_model.predict_proba(sample_df)[0]

print("Prediction:", reloaded_prediction)
print("Class probabilities:", reloaded_probability)


Prediction: 1
Class probabilities: [0.19047619 0.80952381]


If these values match the earlier prediction, the saved model can be reused successfully without retraining.


# 18. Project Workflow Summary

The complete machine-learning workflow is now:

```text
Raw Dataset
     ↓
01 Setup
     ↓
02 Exploratory Data Analysis
     ↓
03 Preprocessing
     ↓
04 Model Training
     ↓
05 Model Evaluation
     ↓
06 Cross-Validation
     ↓
07 Hyperparameter Tuning
     ↓
08 Final Model Selection
     ↓
09 Final Test Evaluation
     ↓
10 Model Interpretation
     ↓
11 Prediction Pipeline
     ↓
Reusable ML Model
```

## Final selected model

```text
StandardScaler
      ↓
KNN
      ├── n_neighbors = 21
      ├── p = 1
      └── weights = uniform
```

## Final unseen-test performance

```text
Accuracy  = 81.97%
Precision = 82.35%
Recall    = 84.85%
F1-score  = 83.58%
ROC-AUC   = 89.45%
```

## Final confusion matrix

```text
[[22, 6],
 [ 5, 28]]
```

Therefore:

```text
TN = 22
FP = 6
FN = 5
TP = 28
```


# 19. Final Conclusion

This notebook transformed the selected KNN model into a reusable prediction pipeline.

The saved Pipeline automatically performs feature scaling before applying the tuned KNN classifier, ensuring that new data follows the same preprocessing procedure used during training.

The pipeline can now:

- Load the saved model
- Accept one new observation
- Predict the target class
- Return class probabilities
- Process multiple observations
- Save prediction results
- Be reloaded later without retraining

The model remains an educational machine-learning system trained on a small dataset. Its predictions should not be treated as medical diagnoses or as a substitute for professional medical evaluation.

## Next Stage

The core ML implementation is now complete.

The remaining work can focus on:

1. Project documentation
2. Results summary
3. Methodology
4. Limitations
5. Final report
6. Optional deployment/demo
